In [4]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import plot

class PlottingMethods:
    """Handles granular chart generation with HTML output wrapping for flexible embedding."""

    @staticmethod
    def _wrap_html(fig):
        """Helper to return HTML representation of a figure."""
        return plot(fig, include_plotlyjs='cdn', output_type='div')

    def generate_bar(self, df, x, y, title="Bar Chart"):
        """Generates a Bar Chart with raw counts and returns HTML."""
        if df is None or df.empty: return "<div>No data</div>"
        fig = px.bar(df, x=x, y=y, title=title, template="plotly_white", text_auto=True)
        return self._wrap_html(fig)

    def generate_histogram(self, df, column, title="Distribution Plot"):
        """Generates a Histogram and returns HTML."""
        if df is None or column not in df.columns: return "<div>No data</div>"
        fig = px.histogram(df, x=column, title=title, template="plotly_white", marginal="box")
        return self._wrap_html(fig)

    def generate_pie(self, df, names, values, title="Proportion Chart"):
        """Generates a Pie Chart and returns HTML."""
        if df is None or names not in df.columns: return "<div>No data</div>"
        fig = px.pie(df, names=names, values=values, title=title)
        return self._wrap_html(fig)

In [5]:
import pandas as pd
import numpy as np
import io
from google.colab import files
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
import plotly.subplots as sp
import scipy.stats as ss

class DataInspector:
    def __init__(self):
        self.df = None
        self.plotter = PlottingMethods()

    def upload_data(self):
        """Handles local file uploads and automatically sanitizes garbage strings."""
        uploaded = files.upload()
        if not uploaded:
            print("⚠️ Upload cancelled by user.")
            return

        filename = list(uploaded.keys())[0]
        # Technical Requirement: Garbage String Handling
        na_vals = ['?', 'n/a', 'NULL', ' ', 'None', 'NA', 'nan', 'N/A']
        self.df = pd.read_csv(io.BytesIO(uploaded[filename]), na_values=na_vals)

        # Technical Requirement: Auto-Type Correction
        self._auto_type_correction()
        print(f"✅ Loaded {filename} successfully.")

    def _auto_type_correction(self):
        """Force-converts columns to numeric if conversion doesn't result in all Nulls."""
        if self.df is None: return
        for col in self.df.columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if not converted.isna().all():
                self.df[col] = converted

    def data_summary(self):
        """Display row/col counts, numeric vs categorical breakdown, and 20-row preview."""
        if self.df is None:
            print("❌ No data available. Please upload a file.")
            return
        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        print("-" * 50)
        print(f"Dataset Size: {self.df.shape[0]} rows x {self.df.shape[1]} columns")
        print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
        print("-" * 50)
        return self.df.head(20)

    def handle_missing_values(self, strategy='mean', fill_value=None):
        """Intelligent Imputation for missing data supporting mean, median, mode, or constant."""
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].isnull().any():
                if strategy == 'mean' and np.issubdtype(self.df[col].dtype, np.number):
                    self.df[col] = self.df[col].fillna(self.df[col].mean())
                elif strategy == 'median' and np.issubdtype(self.df[col].dtype, np.number):
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                elif strategy == 'mode' or not np.issubdtype(self.df[col].dtype, np.number):
                    self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
                elif strategy == 'constant':
                    self.df[col] = self.df[col].fillna(fill_value)
        print(f"✅ Imputation complete using strategy: {strategy}")

    def handle_outliers(self, column, action='flag'):
        """IQR-based outlier detection: flag or automatically delete rows."""
        if self.df is None or column not in self.df.columns: return
        Q1, Q3 = self.df[column].quantile(0.25), self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

        outliers = (self.df[column] < lower) | (self.df[column] > upper)
        if action == 'delete':
            self.df = self.df[~outliers].reset_index(drop=True)
            print(f"✅ Removed outliers from {column}.")
        else:
            self.df[f'{column}_outlier'] = outliers
            print(f"✅ Outliers in {column} flagged in new column.")

    def delete_columns(self):
        """Accepts comma-separated input to prune the dataset."""
        if self.df is None: return
        cols_input = input("Enter column names to delete (comma-separated): ")
        cols = [c.strip() for c in cols_input.split(',')]
        self.df.drop(columns=cols, inplace=True, errors='ignore')
        print(f"Columns remaining: {list(self.df.columns)}")

    def extract_normalized_numeric_data(self, strategy='standard'):
        """Scales numeric data supporting minmax, standard (Z-score), and robust."""
        if self.df is None: return pd.DataFrame()
        num_df = self.df.select_dtypes(include=[np.number])
        scalers = {'standard': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}
        scaled = scalers[strategy].fit_transform(num_df)
        return pd.DataFrame(scaled, columns=num_df.columns)

    def extract_normalized_categorical_data(self, strategy='onehot'):
        """Encodes categorical data supporting onehot, ordinal, and uniform (0-1)."""
        if self.df is None: return pd.DataFrame()
        cat_df = self.df.select_dtypes(exclude=[np.number])
        if strategy == 'onehot':
            return pd.get_dummies(cat_df)
        elif strategy == 'ordinal':
            return pd.DataFrame(OrdinalEncoder().fit_transform(cat_df), columns=cat_df.columns)
        elif strategy == 'uniform':
            encoded = OrdinalEncoder().fit_transform(cat_df)
            return pd.DataFrame(MinMaxScaler().fit_transform(encoded), columns=cat_df.columns)

    def plot_univariate_analysis(self, column):
        """Technical Requirement: 3-panel subplot (Horizontal Violin/Box, Scatter, Histogram)."""
        if self.df is None or column not in self.df.columns: return
        fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Box & Violin", "Index vs Value", "Histogram"))
        fig.add_trace(go.Violin(x=self.df[column], box_visible=True, orientation='h'), row=1, col=1)
        fig.add_trace(go.Scatter(y=self.df[column], mode='markers', marker=dict(opacity=0.4)), row=1, col=2)
        fig.add_trace(go.Histogram(x=self.df[column]), row=1, col=3)
        fig.update_layout(height=400, title_text=f"Univariate Analysis: {column}", showlegend=False)
        fig.show()

    def plot_relationship(self, col1, col2):
        """Technical Requirement: Smart Relationship chart selector (Num-Num, Cat-Num, Cat-Cat)."""
        if self.df is None: return
        is_num1 = np.issubdtype(self.df[col1].dtype, np.number)
        is_num2 = np.issubdtype(self.df[col2].dtype, np.number)

        if is_num1 and is_num2:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title="Num-Num: Scatter with OLS")
        elif not is_num1 and not is_num2:
            fig = px.bar(self.df.groupby([col1, col2]).size().reset_index(name='count'),
                         x=col1, y='count', color=col2, barmode='group', title="Cat-Cat: Grouped Bar")
        else:
            num, cat = (col1, col2) if is_num1 else (col2, col1)
            fig = px.box(self.df, x=cat, y=num, points="all", title="Cat-Num: Box Plot with Points")
        fig.show()

    def plot_all_associations_heatmap(self):
        """Deep Statistical Insight: Heatmap of Numeric Associations."""
        if self.df is None: return
        corr = self.df.select_dtypes(include=[np.number]).corr()
        fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r', title="Association Heatmap (Pearson r)")
        fig.show()

In [7]:
# 1. Initialize
inspector = DataInspector()

# 2. Upload (Select your CSV here)
inspector.upload_data()

# --- ONLY PROCEED IF UPLOAD WAS SUCCESSFUL ---
if inspector.df is not None:
    # 3. Summary
    display(inspector.data_summary())

    # 4. Impute Missing Values
    inspector.handle_missing_values(strategy='median')

    # 5. Normalize (Example: MinMax)
    normalized_df = inspector.extract_normalized_numeric_data(strategy='minmax')
    print("\nNormalized Data Sample:")
    display(normalized_df.head())

    # 6. Advanced Visualization
    # Identify a numeric column for the univariate plot
    num_col = inspector.df.select_dtypes(include=[np.number]).columns[0]
    inspector.plot_univariate_analysis(num_col)

    # Show associations
    inspector.plot_all_associations_heatmap()
else:
    print("Please run this cell again and upload a file to see the analysis.")

⚠️ Upload cancelled by user.
Please run this cell again and upload a file to see the analysis.
